# 02. 질문 2 — 흡연자의 간 수치는 정말 더 높은가? 성별과 음주로 나눠 봐도 같은가?

**자료탐색방법론 기말 팀 프로젝트 · 8조 · 담당: 정형기**

## 질문과 고른 이유
흡연자와 비흡연자의 간 수치(감마지티피)를 그냥 비교하면 흡연자 쪽이 높게 나올 가능성이 큽니다. 그런데 감마지티피는 원래 **남성이 높고 술을 마시는 사람이 높습니다.** 흡연자 중에 남성과 음주자가 많다면, 겉으로 보이는 차이 중 일부는 담배가 아니라 성별과 음주 때문일 수 있습니다. 그래서 성별과 음주 여부가 같은 사람끼리 나눠서(층화) 다시 비교합니다.

## 분석 순서
1. 흡연 집단의 구성 비교: 흡연자에 남성·음주자가 정말 많은가? (혼재변수 후보 확인)
2. 전체 비교: 흡연 상태별 감마지티피
3. 성별로 나눠 비교
4. 성별 × 음주로 나눠 비교 (핵심)
5. 간 수치 이상 비율로 다시 확인 (비율의 분모 명시)
6. 연령대로도 나눠 보기 (남은 혼재변수 점검)
7. 주요 결과 3건 검증 → 검증 로그
8. 결론과 한계

**요약값:** 감마지티피는 오른쪽으로 길게 늘어진 분포라(질문 1 참고) 평균 대신 **중앙값**과 사분위수를 씁니다. 그림은 `log10` 눈금을 씁니다.

## 0. 준비: 데이터 불러오기와 한글 글꼴

In [ ]:
import os, sys, zipfile, urllib.request, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# ---- 한글 글꼴 (Colab에서는 나눔고딕 설치) ----
def set_korean_font():
    try:
        import google.colab  # noqa
        subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"], check=False,
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except ImportError:
        pass
    others = sorted(fm.findSystemFonts(), key=lambda p: ("Sans" not in p, p))   # 고딕(Sans) 계열 우선
    for path in ["/usr/share/fonts/truetype/nanum/NanumGothic.ttf"] + others:
        if os.path.exists(path) and ("Nanum" in path or "CJK" in path or "Malgun" in path or "AppleGothic" in path):
            fm.fontManager.addfont(path)
            plt.rcParams["font.family"] = fm.FontProperties(fname=path).get_name()
            break
    plt.rcParams["axes.unicode_minus"] = False
set_korean_font()
plt.rcParams["figure.dpi"] = 110

# ---- 데이터: 00_데이터정리.ipynb와 같은 규칙으로 만든다 ----
ZIP_NAME = "hc2023.zip"
RAW_URL = "https://github.com/wjaioajqdo12-coder/eda-team8-smoking/releases/download/data/hc2023.zip"

def load_raw():
    if not os.path.exists(ZIP_NAME):
        urllib.request.urlretrieve(RAW_URL, ZIP_NAME)
    with zipfile.ZipFile(ZIP_NAME) as zf:
        member = [m for m in zf.namelist() if m.lower().endswith(".csv")][0]
        with zf.open(member) as f:
            return pd.read_csv(f, encoding="cp949")

raw = load_raw()
SEX = {1: "남", 2: "여"}
SMK = {1: "비흡연", 2: "과거 흡연", 3: "현재 흡연"}
DRK = {0: "비음주", 1: "음주"}
AGE = {c: (f"{(c-1)*5}~{(c-1)*5+4}세" if c < 18 else "85세 이상") for c in range(5, 19)}

df = pd.DataFrame({
    "id": raw["가입자일련번호"],
    "성별": raw["성별코드"].map(SEX),
    "연령대": pd.Categorical(raw["연령대코드(5세단위)"].map(AGE), categories=list(AGE.values()), ordered=True),
    "감마지티피": raw["감마지티피"],
    "흡연": pd.Categorical(raw["흡연상태"].map(SMK), categories=list(SMK.values()), ordered=True),
    "음주": pd.Categorical(raw["음주여부"].map(DRK), categories=list(DRK.values())),
})
print("원자료:", raw.shape)

# 질문 2에 필요한 열에서만 결측 제외 (00 노트북의 원칙)
cols = ["감마지티피", "흡연", "성별", "음주"]
q2 = df.dropna(subset=cols).copy()
q2["log10_ggt"] = np.log10(q2["감마지티피"])
print(f"분석 대상: {len(q2):,}명 (제외 {len(df)-len(q2):,}명: {cols} 중 하나라도 결측)")

## 1. 흡연 집단의 구성은 같은가? (혼재변수 후보 확인)
층화가 필요한 이유를 먼저 숫자로 보입니다. 흡연 집단마다 남성 비율과 음주자 비율이 다르면, 성별과 음주가 혼재변수가 될 수 있습니다.

In [ ]:
comp = pd.DataFrame({
    "인원": q2.groupby("흡연", observed=True).size(),
    "남성 비율(%)": q2.groupby("흡연", observed=True)["성별"].apply(lambda s: (s == "남").mean() * 100),
    "음주자 비율(%)": q2.groupby("흡연", observed=True)["음주"].apply(lambda s: (s == "음주").mean() * 100),
}).round(1)
comp

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.6))
x = np.arange(len(comp))
ax.bar(x - 0.2, comp["남성 비율(%)"], 0.4, label="남성 비율", color="#4C78A8")
ax.bar(x + 0.2, comp["음주자 비율(%)"], 0.4, label="음주자 비율", color="#F58518")
for i, (m, d) in enumerate(zip(comp["남성 비율(%)"], comp["음주자 비율(%)"])):
    ax.text(i - 0.2, m + 1.5, f"{m:.0f}%", ha="center", fontsize=9)
    ax.text(i + 0.2, d + 1.5, f"{d:.0f}%", ha="center", fontsize=9)
ax.set_xticks(x, comp.index)
ax.set_ylim(0, 105); ax.set_ylabel("비율 (%)")
ax.set_title("흡연 집단별 남성·음주자 비율 (분모: 각 흡연 집단 인원)")
ax.legend(frameon=False, loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig("fig_q2_1_구성.png", dpi=200); plt.show()

In [ ]:
# 나이도 혼재변수일 수 있는지: 흡연 집단별 연령대 중앙값
age_code = raw.loc[q2.index, "연령대코드(5세단위)"]
print("흡연 집단별 연령대 중앙값:", {k: AGE[int(v)] for k, v in age_code.groupby(q2["흡연"], observed=True).median().items()})

**읽는 법:** 현재 흡연자는 약 90%가 남성이고 84%가 음주자입니다. 비흡연자는 남성이 30%, 음주자가 57%입니다. 흡연 집단끼리 성별·음주 구성이 크게 다르므로 단순 비교는 담배의 영향과 성별·음주의 영향이 섞인 결과입니다. 연령대 중앙값은 세 집단이 비슷해서, 나이는 성별·음주보다 영향이 작을 것으로 예상합니다(6번에서 확인).

## 2. 전체 비교: 흡연 상태별 감마지티피
질문 2 전체에서 쓰는 요약 함수를 하나 만들어 둡니다. 7번 검증에서 이 함수에 시험 입력을 넣어 확인합니다.

In [ ]:
def summarize(data, by, value="감마지티피"):
    """집단별 인원과 5수요약(중앙값 중심). 결측은 미리 뺀 자료를 넣는다."""
    g = data.groupby(by, observed=True)[value]
    out = pd.DataFrame({
        "인원": g.size(),
        "최솟값": g.min(),
        "Q1": g.quantile(0.25),
        "중앙값": g.median(),
        "Q3": g.quantile(0.75),
        "최댓값": g.max(),
    })
    return out

overall = summarize(q2, "흡연")
overall

In [ ]:
def boxplot_by(data, groups, ax, title):
    vals = [data.loc[data["흡연"] == g, "log10_ggt"] for g in groups]
    ax.boxplot(vals, tick_labels=groups, showfliers=False, widths=0.5,
               medianprops=dict(color="#E45756", linewidth=2))
    ticks = [10, 20, 30, 50, 100, 200]
    ax.set_yticks(np.log10(ticks), [str(t) for t in ticks])
    ax.set_ylabel("감마지티피 (IU/L, log 눈금)")
    ax.set_title(title)
    ax.spines[["top", "right"]].set_visible(False)

fig, ax = plt.subplots(figsize=(5.5, 4))
boxplot_by(q2, list(SMK.values()), ax, "전체: 흡연 상태별 감마지티피")
for i, g in enumerate(SMK.values(), start=1):
    ax.text(i + 0.28, np.log10(overall.loc[g, "중앙값"]), f"{overall.loc[g, '중앙값']:.0f}", va="center", fontsize=9)
plt.tight_layout(); plt.savefig("fig_q2_2_전체.png", dpi=200); plt.show()
print("상자 밖 점(극단값)은 100만 명이라 너무 많아 생략했습니다. 상자는 Q1~Q3, 빨간 선은 중앙값입니다.")

In [ ]:
gap_overall = overall.loc["현재 흡연", "중앙값"] - overall.loc["비흡연", "중앙값"]
print(f"전체: 현재 흡연 - 비흡연 중앙값 차이 = {gap_overall:.0f} IU/L")

## 3. 성별로 나눠 비교

In [ ]:
by_sex = summarize(q2, ["성별", "흡연"])[["인원", "Q1", "중앙값", "Q3"]]
by_sex

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8), sharey=True)
for ax, s in zip(axes, ["남", "여"]):
    boxplot_by(q2[q2["성별"] == s], list(SMK.values()), ax, f"{s}성")
axes[1].set_ylabel("")
plt.suptitle("성별로 나눈 흡연 상태별 감마지티피")
plt.tight_layout(); plt.savefig("fig_q2_3_성별.png", dpi=200); plt.show()

med_sex = by_sex["중앙값"].unstack("흡연")
med_sex["차이(현재-비흡연)"] = med_sex["현재 흡연"] - med_sex["비흡연"]
med_sex

## 4. 성별 × 음주로 나눠 비교 (핵심)
성별과 음주 여부가 같은 사람끼리 4개 층으로 나눕니다. 층마다 인원을 함께 적습니다.

In [ ]:
strata = summarize(q2, ["성별", "음주", "흡연"])
med = strata["중앙값"].unstack("흡연")
n = strata["인원"].unstack("흡연")
table4 = med.copy()
table4["차이(현재-비흡연)"] = med["현재 흡연"] - med["비흡연"]
table4["인원(비흡연/과거/현재)"] = n.apply(lambda r: " / ".join(f"{int(v):,}" for v in r), axis=1)
table4

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
labels = ["전체"] + [f"{s}·{d}" for s, d in table4.index]
never = [overall.loc["비흡연", "중앙값"]] + list(table4["비흡연"])
cur = [overall.loc["현재 흡연", "중앙값"]] + list(table4["현재 흡연"])
y = np.arange(len(labels))[::-1]
for yi, a, b in zip(y, never, cur):
    ax.plot([a, b], [yi, yi], color="#BBBBBB", linewidth=3, zorder=1)
    ax.text(b + 0.6, yi, f"차이 {b - a:.0f}", va="center", fontsize=10)
ax.scatter(never, y, color="#4C78A8", s=70, label="비흡연", zorder=2)
ax.scatter(cur, y, color="#E45756", s=70, label="현재 흡연", zorder=2)
ax.axhline(y[0] - 0.5, color="#DDDDDD", linewidth=1)
ax.set_yticks(y, labels)
ax.set_xlabel("감마지티피 중앙값 (IU/L)")
ax.set_xlim(10, 45)
ax.set_title("전체로 보면 차이 15 → 성별·음주가 같은 사람끼리는 2~11")
ax.legend(frameon=False, loc="lower right")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig("fig_q2_4_층화핵심.png", dpi=200); plt.show()

**읽는 법:**
- 전체로 보면 현재 흡연자의 중앙값이 비흡연자보다 15 높습니다.
- 성별과 음주 여부가 같은 사람끼리 비교하면 차이는 2~11로 줄어듭니다. 전체 차이의 상당 부분은 **흡연자 중에 남성과 음주자가 많아서** 생긴 것입니다.
- 그래도 네 층 모두에서 현재 흡연자가 더 높습니다. 방향이 뒤집히지는 않았으므로 심슨의 역설(방향 역전)은 아니고, **혼재변수 때문에 차이가 부풀려진 경우**입니다.
- 차이는 **술을 마시는 층**에서 큽니다(남성 11, 여성 6). 술을 마시지 않는 층에서는 3, 2로 작습니다. 흡연과 간 수치의 관계가 음주 여부에 따라 다르게 나타납니다.

## 5. 간 수치 이상 비율로 다시 확인
중앙값 대신 "정상 범위를 넘는 사람의 비율"로 같은 비교를 합니다. 매뉴얼의 정상치(남성 63, 여성 35 IU/L 이하)를 기준으로 씁니다. **분모는 각 칸의 인원**입니다.

In [ ]:
q2["이상"] = np.where(q2["성별"] == "남", q2["감마지티피"] > 63, q2["감마지티피"] > 35)

def rate_table(data, by):
    g = data.groupby(by, observed=True)["이상"]
    t = pd.DataFrame({"이상(분자)": g.sum(), "인원(분모)": g.size()})
    t["이상 비율(%)"] = (t["이상(분자)"] / t["인원(분모)"] * 100).round(1)
    return t

rate_all = rate_table(q2, "흡연")
rate_all

In [ ]:
rate_strata = rate_table(q2, ["성별", "음주", "흡연"])["이상 비율(%)"].unstack("흡연")
rate_strata["차이(%p)"] = rate_strata["현재 흡연"] - rate_strata["비흡연"]
rate_strata

**읽는 법:**
- 전체로는 이상 비율이 11.6% → 23.7%로 12.1%p 차이 납니다.
- 술을 마시지 않는 층에서는 차이가 작고(남 1.5%p, 여 4.4%p), 술을 마시는 층에서는 큽니다(남 12.8%p, 여 14.1%p).
- 중앙값과 달리 음주 층의 차이가 전체 차이보다 줄지 않았습니다. 이상 비율은 **남녀 정상치를 따로 써서 이미 성별 차이를 어느 정도 걸러낸 지표**라, 성별 때문에 부풀려지는 부분이 처음부터 작기 때문입니다.
- 두 지표가 공통으로 보여 주는 것은 **"흡연과 간 수치의 관계는 술을 마시는 사람에게서 크게 나타난다"**는 점입니다.

## 6. 연령대로도 나눠 보기
1번에서 흡연 집단의 연령대 중앙값은 비슷했지만, 나이에 따라 차이가 달라지는지 확인합니다. 성별 × 음주 층 안에서 다시 연령대별로 나눕니다.

In [ ]:
age_gap = (q2.groupby(["성별", "음주", "연령대", "흡연"], observed=True)["감마지티피"].median()
             .unstack("흡연"))
age_gap["차이"] = age_gap["현재 흡연"] - age_gap["비흡연"]
age_n = q2[q2["흡연"] == "현재 흡연"].groupby(["성별", "음주", "연령대"], observed=True).size()

fig, ax = plt.subplots(figsize=(8.5, 4))
colors = {("남", "음주"): "#E45756", ("남", "비음주"): "#F2A7A6", ("여", "음주"): "#4C78A8", ("여", "비음주"): "#9ECAE9"}
for key, c in colors.items():
    s = age_gap.loc[key, "차이"]
    ok = age_n.loc[key].reindex(s.index) >= 100      # 현재 흡연자가 100명 미만인 칸은 불안정해서 뺌
    ax.plot(np.arange(len(s))[ok.values], s[ok].values, marker="o", color=c, label=f"{key[0]}·{key[1]}")
ax.axhline(0, color="#999999", linewidth=1)
ax.set_xticks(np.arange(len(AGE)), list(AGE.values()), rotation=45, ha="right", fontsize=8)
ax.set_ylabel("중앙값 차이 (현재 흡연 - 비흡연)")
ax.set_title("연령대별로 나눠도 대부분 흡연자가 더 높음 (현재 흡연자 100명 이상인 칸만)")
ax.legend(frameon=False, ncol=4, fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig("fig_q2_6_연령대.png", dpi=200); plt.show()

age_gap.loc[("남", "음주"), ["비흡연", "현재 흡연", "차이"]].T

**읽는 법:** 성별·음주·연령대까지 같은 사람끼리 비교해도 대부분의 칸에서 현재 흡연자가 더 높습니다. 술을 마시는 남성에서는 30~40대에 차이가 가장 크고(13 안팎), 나이가 많아질수록 줄어듭니다. 나이 때문에 결론이 바뀌지는 않습니다.

## 참고: 감마지티피 9999(13명)를 빼도 같은가?
00 노트북에서 9999는 실제 높은 값의 상한 표기로 보고 남겨 두었습니다. 중앙값이 이 13명에 흔들리지 않는지 확인합니다.

In [ ]:
without = q2[q2["감마지티피"] != 9999]
cmp9999 = pd.DataFrame({
    "포함": q2.groupby(["성별", "음주", "흡연"], observed=True)["감마지티피"].median(),
    "9999 제외": without.groupby(["성별", "음주", "흡연"], observed=True)["감마지티피"].median(),
})
print("9999 인원:", (q2["감마지티피"] == 9999).sum(), "명 / 중앙값이 바뀐 칸 수:", (cmp9999["포함"] != cmp9999["9999 제외"]).sum())

## 7. 주요 결과 3건 검증
각 검증은 **실행 전에 예상값을 적고**, 확인 절차를 실행한 뒤 실제 결과와 비교합니다.

### 검증 1 — 손계산 대조: 층별 중앙값
- **대상 결과:** 4번 표의 층별 중앙값 계산이 맞는가
- **예상값:** 아래에서 뽑은 7명의 감마지티피를 손으로 정렬하면 가운데(4번째) 값이 중앙값이고, 코드의 `median()`과 같다
- **확인 절차:** 술을 마시는 남성 현재 흡연자 중 7명을 무작위로 뽑아(`random_state=8`) 값을 출력하고, 표 아래에서 손으로 정렬해 중앙값을 구한 뒤 코드 결과와 비교

In [ ]:
pick = q2[(q2["성별"] == "남") & (q2["음주"] == "음주") & (q2["흡연"] == "현재 흡연")].sample(7, random_state=8)
print("뽑은 7명의 감마지티피 (뽑힌 순서):", pick["감마지티피"].astype(int).tolist())
print("코드로 구한 중앙값:", pick["감마지티피"].median())

**손계산:** 뽑힌 값 125, 26, 45, 12, 34, 33, 67을 작은 순서로 다시 쓰면 12, 26, 33, **34**, 45, 67, 125이고, 7개의 가운데인 4번째 값은 **34**입니다. 코드 결과 34.0과 같습니다. (발표 질의응답에 대비해 팀원도 한 번 직접 해 보세요. 같은 방법으로 "비흡연" 7명도 뽑아 확인할 수 있습니다.)

### 검증 2 — 다른 계산 경로로 교차확인: 핵심 표를 원자료 코드에서 다시 계산
- **대상 결과:** 4번 핵심 표(성별 × 음주 × 흡연 중앙값)와 1번 구성 비율
- **예상값:** 글자 변환·범주 처리를 전혀 거치지 않고 원자료의 숫자 코드만으로 다시 계산해도 12개 중앙값과 남성 비율이 모두 같다
- **확인 절차:** `raw`에서 코드 열만 꺼내 결측을 빼고, 코드 기준으로 묶어 중앙값과 비율을 다시 계산한 뒤 4번·1번 결과와 하나씩 비교. 다르면 코드를 글자로 바꾸는 과정이나 결측 처리에 실수가 있다는 뜻

In [ ]:
r = raw[["성별코드", "음주여부", "흡연상태", "감마지티피"]].dropna()
alt = r.groupby(["성별코드", "음주여부", "흡연상태"])["감마지티피"].median()
alt.index = pd.MultiIndex.from_tuples([(SEX[a], DRK[int(b)], SMK[int(c)]) for a, b, c in alt.index], names=["성별", "음주", "흡연"])
main = strata["중앙값"]
both = pd.DataFrame({"주 경로(4번)": main, "원자료 코드 경로": alt.reindex(main.index)})
both["같음"] = both["주 경로(4번)"] == both["원자료 코드 경로"]
print("분석 인원 비교:", len(q2), "vs", len(r))
alt_male = r.groupby("흡연상태")["성별코드"].apply(lambda s: round((s == 1).mean() * 100, 1))
print("남성 비율(원자료 코드 경로):", dict(zip(SMK.values(), alt_male.values)), "/ 1번 표:", comp["남성 비율(%)"].tolist())
both

### 검증 3 — 시험 입력: 결측·극단값·1명짜리 집단
- **대상 결과:** 2·4번에서 쓴 `summarize` 함수와 결측 제외 방식
- **예상값 (실행 전에 손으로 정함):**
  - 흡연이 비어 있는 사람(값 500)은 분석에서 빠져야 한다 → 전체 인원 7명
  - 비흡연 3명(10, 20, 30) → 중앙값 20
  - 과거 흡연 1명(40) → 인원 1, 중앙값 40 (1명짜리 집단도 오류 없이 계산)
  - 현재 흡연 3명(20, 30, 9999) → 중앙값 30 (9999가 있어도 중앙값은 흔들리지 않음), 최댓값 9999
- **확인 절차:** 가짜 자료 8행을 만들어 본 분석과 똑같은 순서(결측 제외 → `summarize`)로 넣고 결과를 예상값과 비교

In [ ]:
fake = pd.DataFrame({
    "감마지티피": [10, 20, 30, 40, 20, 30, 9999, 500],
    "흡연": pd.Categorical(["비흡연", "비흡연", "비흡연", "과거 흡연", "현재 흡연", "현재 흡연", "현재 흡연", None],
                         categories=list(SMK.values()), ordered=True),
    "성별": ["남"] * 8, "음주": ["음주"] * 8,
})
fake_q = fake.dropna(subset=cols)
res = summarize(fake_q, "흡연")
print("분석 인원:", len(fake_q), "(예상 7)")
expected = {"비흡연": (3, 20), "과거 흡연": (1, 40), "현재 흡연": (3, 30)}
for g, (n_exp, med_exp) in expected.items():
    ok = (res.loc[g, "인원"] == n_exp) and (res.loc[g, "중앙값"] == med_exp)
    print(f"{g}: 인원 {res.loc[g, '인원']} (예상 {n_exp}), 중앙값 {res.loc[g, '중앙값']} (예상 {med_exp}) → {'일치' if ok else '불일치'}")
print("현재 흡연 최댓값:", res.loc["현재 흡연", "최댓값"], "(예상 9999)")
res

## 검증 로그 (보고서·부록에 옮겨 쓰기)

| # | 대상 결과 | 예상값 | 확인 절차 | 실제 결과 | 불일치와 수정 |
|---|---|---|---|---|---|
| 1 | 층별 중앙값(4번) | 7명 손 정렬 4번째 값 = 코드 중앙값 | 남·음주·현재 흡연 7명 무작위 추출(`random_state=8`), 손으로 정렬 | 뽑힌 값 125, 26, 45, 12, 34, 33, 67 → 정렬 12, 26, 33, **34**, 45, 67, 125 → 손계산 34 = 코드 34 | 없음 |
| 2 | 핵심 표 12개 중앙값, 남성 비율(1·4번) | 원자료 코드만으로 다시 계산해도 모두 같음 | 글자 변환 없이 원자료 코드로 재계산 후 비교 | 12개 모두 같음, 분석 인원 994,054명 같음, 남성 비율 같음 | 없음 |
| 3 | `summarize` 함수와 결측 제외 | 인원 7, 중앙값 20 / 40 / 30, 최댓값 9999 | 결측·9999·1명 집단이 든 가짜 자료 8행 투입 | 모두 예상과 일치 | 없음 |
| 참고 | 9999 처리(00 노트북) | 결측 표시일 것 | 9999인 13명의 AST·ALT 확인 | 13명 모두 AST 또는 ALT가 정상치 초과 | 결측 처리하지 않고 유지·표시. 제외해도 층별 중앙값 변화 없음 |

## 8. 결론

**질문:** 흡연자의 간 수치가 비흡연자보다 높은가? 성별과 음주 여부로 나눠 봐도 같은가?

1. **전체로 보면** 현재 흡연자의 감마지티피 중앙값은 34로 비흡연자(19)보다 15 높다. 이상 비율도 23.7% 대 11.6%로 두 배 정도다.
2. **그러나 흡연자 집단은 구성이 다르다.** 현재 흡연자의 약 90%가 남성, 84%가 음주자로, 비흡연자(남성 30%, 음주자 57%)와 크게 다르다.
3. **성별과 음주가 같은 사람끼리 비교하면** 차이는 2~11로 줄어든다. 전체 차이의 상당 부분은 성별과 음주 구성 차이에서 온 것이다.
4. **그래도 차이는 사라지지 않는다.** 네 층 모두, 그리고 연령대까지 나눈 대부분의 칸에서 현재 흡연자가 더 높다.
5. **차이는 술을 마시는 사람에게서 크다.** 중앙값 차이는 음주 층 11(남)·6(여), 비음주 층 3·2이다. 정상치를 넘는 비율로 봐도 음주 층 12.8·14.1%p, 비음주 층 1.5·4.4%p로 같은 경향이다.

**한 줄 요약:** 흡연자의 간 수치가 높아 보이는 이유의 상당 부분은 성별과 음주 때문이지만, 같은 조건끼리 비교해도 흡연자 쪽이 조금 더 높다.

## 한계 (해석할 때 꼭 함께 말할 것)
- **관계이지 원인이 아니다.** 한 시점에 조사한 자료라서 담배가 간 수치를 올렸다고 말할 수 없다.
- **음주는 양이 아니라 여부만 있다.** 같은 "음주" 층 안에서도 흡연자가 술을 더 많이 마실 수 있다. 음주 층에서 차이가 큰 것은 음주량 차이 때문일 가능성도 있어서, 남은 차이를 모두 담배 탓으로 볼 수 없다.
- **흡연·음주는 본인 응답이다.** 실제보다 적게 답했을 수 있다.
- **다른 혼재변수(체중, 복용 약, 간 질환 등)는 이번 분석에서 다루지 않았다.**

## 발표용 그림 파일
이 노트북을 실행하면 같은 폴더에 PPT에 넣을 그림이 저장됩니다(Colab 왼쪽 폴더 아이콘에서 내려받기).
- `fig_q2_1_구성.png` — 흡연 집단별 남성·음주자 비율
- `fig_q2_2_전체.png` — 전체 비교 상자그림
- `fig_q2_3_성별.png` — 성별로 나눈 상자그림
- `fig_q2_4_층화핵심.png` — **핵심 그림:** 전체 차이 15 → 층별 2~11
- `fig_q2_6_연령대.png` — 연령대별 차이